# Phase 1: Setup & Data Loading
## Harvard Healthy Minds Study (HMS) — 2023-2024 & 2024-2025

**Dataset:** Healthy Minds Study — one of the largest annual surveys of college student mental health in the United States.  
**Source:** [Healthy Minds Network](https://healthymindsnetwork.org/)  
**Years:** 2023-2024 (104,729 students) and 2024-2025 (84,735 students)  
**Institutions:** Hundreds of US colleges and universities

### Key Instruments
| Instrument | Measures | Items | Scale |
|-----------|----------|-------|-------|
| PHQ-9 | Depression severity | 9 | 0–3 per item |
| GAD-7 | Anxiety severity | 7 | 0–3 per item |
| Diener Flourishing Scale | Psychological well-being | 8 | 1–7 per item |
| UCLA Loneliness (3-item) | Loneliness | 3 | 1–3 per item |

### Research Questions
1. How prevalent are depression, anxiety, and suicidality among US college students?
2. Did mental health improve or worsen from 2023-24 to 2024-25?
3. Which demographic groups are most at risk?
4. How do institutional characteristics (size, type, public/private) relate to student mental health?
5. What is the relationship between financial stress, loneliness, and mental health outcomes?
6. What are the barriers to seeking mental health treatment?


## 1.1 Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1.2 Load Both Years

In [2]:
df23 = pd.read_csv('HMS_2023-2024_PUBLIC_instchars.csv', low_memory=False)
df24 = pd.read_csv('HMS_2024-2025_PUBLIC_instchars.csv', low_memory=False)

print(f"2023-2024: {df23.shape[0]:,} students, {df23.shape[1]:,} columns")
print(f"2024-2025: {df24.shape[0]:,} students, {df24.shape[1]:,} columns")
print(f"Common columns: {len(set(df23.columns) & set(df24.columns)):,}")

2023-2024: 104,729 students, 1,554 columns
2024-2025: 84,735 students, 1,608 columns
Common columns: 1,378


## 1.3 Select Relevant Columns

In [3]:
# Column definitions
DEMO_COLS = ['age', 'sex_birth', 'gender_male', 'gender_female', 'gender_queer', 'gender_nonbin',
             'race_black', 'race_white', 'race_asian', 'race_his', 'race_ainaan',
             'international', 'pellgrant', 'housing_worry',
             'degree_bach', 'degree_ma', 'degree_phd',
             'yr_sch', 'enroll']

INST_COLS = ['inst_type', 'inst_public', 'inst_size', 'inst_geo']

PHQ9_ITEMS = [f'phq9_{i}' for i in range(1, 10)]
GAD7_ITEMS = [f'gad7_{i}' for i in range(1, 8)]
DIENER_ITEMS = [f'diener{i}' for i in range(1, 9)]
LONELY_ITEMS = ['lone_lackcompanion', 'lone_leftout', 'lone_isolated']

OUTCOME_COLS = ['dep_any', 'dep_maj', 'anx_any', 'anx_sev',
                'sui_idea', 'sui_plan', 'sui_att',
                'deprawsc', 'anx_score', 'flourish', 'lonely',
                'sib_any', 'sub_any', 'ed_any']

TREATMENT_COLS = ['ther_any', 'meds_any', 'percneed', 'percneed_cur']

STRESS_COLS = ['housing_worry', 'food_worry', 'FinStress']

ALL_COLS = list(set(DEMO_COLS + INST_COLS + PHQ9_ITEMS + GAD7_ITEMS +
                    DIENER_ITEMS + LONELY_ITEMS + OUTCOME_COLS +
                    TREATMENT_COLS + STRESS_COLS))

# Keep only columns present in both datasets
common_cols = list(set(df23.columns) & set(df24.columns))
keep = [c for c in ALL_COLS if c in common_cols]

df23_sel = df23[keep].copy()
df24_sel = df24[keep].copy()

df23_sel['year'] = '2023-2024'
df24_sel['year'] = '2024-2025'

print(f"Selected {len(keep)} columns present in both years")
print(f"2023-2024 subset: {df23_sel.shape}")
print(f"2024-2025 subset: {df24_sel.shape}")

Selected 70 columns present in both years
2023-2024 subset: (104729, 71)
2024-2025 subset: (84735, 71)


## 1.4 Combine & Preview

In [4]:
df = pd.concat([df23_sel, df24_sel], ignore_index=True)
print(f"Combined dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("Sample (5 rows):")
df[['year','age','dep_any','anx_any','sui_idea','deprawsc','anx_score']].head()

Combined dataset: 189,464 rows x 71 columns

Sample (5 rows):


,year,age,dep_any,anx_any,sui_idea,deprawsc,anx_score
0,2023-2024,18.0,1.0,0.0,0.0,17.0,7.0
1,2023-2024,18.0,0.0,0.0,0.0,4.0,5.0
2,2023-2024,18.0,0.0,1.0,0.0,8.0,10.0
3,2023-2024,20.0,1.0,0.0,0.0,12.0,8.0
4,2023-2024,21.0,0.0,0.0,0.0,5.0,6.0


## 1.5 Column Reference

In [5]:
reference = {
    'age': 'Student age (years)',
    'sex_birth': 'Sex assigned at birth (1=Male, 2=Female, 3=Intersex)',
    'gender_male': 'Identifies as male (1=yes)',
    'gender_female': 'Identifies as female (1=yes)',
    'gender_queer': 'Identifies as genderqueer (1=yes)',
    'gender_nonbin': 'Identifies as non-binary (1=yes)',
    'race_black': 'Black/African American (1=yes)',
    'race_white': 'White (1=yes)',
    'race_asian': 'Asian (1=yes)',
    'race_his': 'Hispanic/Latino (1=yes)',
    'international': 'International student (1=yes)',
    'pellgrant': 'Pell grant recipient — low income proxy (1=yes)',
    'housing_worry': 'Housing insecurity (1=no worry, 2=some, 3=high)',
    'yr_sch': 'Year in school (1=1st year ... 5=5th+)',
    'inst_type': 'Institution type (1=2yr, 2=4yr private, 3=4yr public, 4=grad, 5=other)',
    'inst_public': 'Public institution (1=yes)',
    'inst_size': 'Institution size (1=<1k, 2=1-5k, 3=5-10k, 4=10-20k, 5=>20k)',
    'inst_geo': 'Geographic region (1-9 US census regions)',
    'dep_any': 'Any depression (PHQ-9 based, 1=yes)',
    'dep_maj': 'Major depression (PHQ-9 >= 10, 1=yes)',
    'anx_any': 'Any anxiety (GAD-7 based, 1=yes)',
    'anx_sev': 'Severe anxiety (GAD-7 >= 15, 1=yes)',
    'sui_idea': 'Suicidal ideation past year (1=yes)',
    'sui_plan': 'Suicide plan past year (1=yes)',
    'sui_att': 'Suicide attempt past year (1=yes)',
    'deprawsc': 'PHQ-9 raw score (0-27)',
    'anx_score': 'GAD-7 raw score (0-21)',
    'flourish': 'Diener Flourishing Scale score (8-56)',
    'lonely': 'UCLA Loneliness score (3-9)',
    'ther_any': 'Currently in therapy (1=yes)',
    'meds_any': 'Currently on psychiatric medication (1=yes)',
    'percneed': 'Perceives need for mental health help (1=yes)',
    'percneed_cur': 'Currently perceives unmet need (1=yes)',
    'sib_any': 'Non-suicidal self-injury (1=yes)',
    'sub_any': 'Any substance use (1=yes)',
    'food_worry': 'Food insecurity (1=yes)',
    'FinStress': 'Financial stress score',
}
ref_df = pd.DataFrame(list(reference.items()), columns=['Column','Description'])
ref_df[ref_df['Column'].isin(keep)]

,Column,Description
0,age,Student age (years)
1,sex_birth,"Sex assigned at birth (1=Male, 2=Female, 3=Int..."
2,gender_male,Identifies as male (1=yes)
3,gender_female,Identifies as female (1=yes)
4,gender_queer,Identifies as genderqueer (1=yes)
5,gender_nonbin,Identifies as non-binary (1=yes)
6,race_black,Black/African American (1=yes)
7,race_white,White (1=yes)
8,race_asian,Asian (1=yes)
9,race_his,Hispanic/Latino (1=yes)
